In [1]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN,Dense

In [4]:
!pip install --upgrade certifi

  Attempting uninstall: certifi
    Found existing installation: certifi 2026.6.17
    Uninstalling certifi-2026.6.17:
      Successfully uninstalled certifi-2026.6.17


In [9]:
import os
os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()

In [11]:
import os
import ssl
import certifi
import urllib.request
import numpy as np

# Build an SSL context that ONLY uses certifi's cert bundle,
# completely avoiding the Windows certificate store
ctx = ssl.create_default_context(cafile=certifi.where())

url = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/imdb.npz"
save_path = "imdb.npz"

if not os.path.exists(save_path):
    with urllib.request.urlopen(url, context=ctx) as response, open(save_path, "wb") as out_file:
        out_file.write(response.read())

with np.load(save_path, allow_pickle=True) as f:
    X_train, y_train = f["x_train"], f["y_train"]
    X_test, y_test = f["x_test"], f["y_test"]

print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(25000,) (25000,) (25000,) (25000,)


In [19]:
import numpy as np

num_words = 10000

with np.load("imdb.npz", allow_pickle=True) as f:
    X_train, y_train = f["x_train"], f["y_train"]
    X_test, y_test = f["x_test"], f["y_test"]

def cap_vocab(sequences, num_words):
    # Replace any index >= num_words with 2, the standard Keras OOV token
    return np.array([
        [w if w < num_words else 2 for w in seq]
        for seq in sequences
    ], dtype=object)

X_train = cap_vocab(X_train, num_words)
X_test = cap_vocab(X_test, num_words)

In [20]:
from keras.preprocessing.sequence import pad_sequences

# Pad sequences to have the same length
X_train = pad_sequences(X_train, maxlen=100)
X_test = pad_sequences(X_test, maxlen=100)

print(X_train.shape, X_test.shape)

(25000, 100) (25000, 100)


In [21]:
from keras.layers import Input

model = Sequential([
    Input(shape=(100,)),
    Embedding(input_dim=10000, output_dim=32),
    SimpleRNN(5, return_sequences=True),
    SimpleRNN(5),
    Dense(1, activation='sigmoid')
])

model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 100, 32)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_8 (SimpleRNN)        │ (None, 100, 5)         │           190 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_9 (SimpleRNN)        │ (None, 5)              │            55 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │             6 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 320,251 (1.22 MB)

 Trainable params: 320,251 (1.22 MB)

 Non-trainable params: 0 (0.00 B)

In [22]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [23]:
# train model
history=model.fit(X_train,y_train,epochs=20,batch_size=32,validation_split=0.2)

Epoch 1/20


625/625 ━━━━━━━━━━━━━━━━━━━━ 15s 21ms/step - accuracy: 0.6802 - loss: 0.6046 - val_accuracy: 0.6854 - val_loss: 0.6697
Epoch 2/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 13s 21ms/step - accuracy: 0.7984 - loss: 0.4456 - val_accuracy: 0.6428 - val_loss: 0.8261
Epoch 3/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 14s 22ms/step - accuracy: 0.8469 - loss: 0.3627 - val_accuracy: 0.7046 - val_loss: 0.7149
Epoch 4/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 14s 22ms/step - accuracy: 0.9133 - loss: 0.2420 - val_accuracy: 0.6460 - val_loss: 0.8977
Epoch 5/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 15s 24ms/step - accuracy: 0.9494 - loss: 0.1605 - val_accuracy: 0.7024 - val_loss: 0.8219
Epoch 6/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 15s 24ms/step - accuracy: 0.9667 - loss: 0.1096 - val_accuracy: 0.7466 - val_loss: 0.7742
Epoch 7/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 15s 24ms/step - accuracy: 0.9786 - loss: 0.0742 - val_accuracy: 0.6956 - val_loss: 1.0532
Epoch 8/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 15s 23ms/step - accuracy: 0.9801 - loss: 0.0650 - val_accurac